# 🎯 手写 PPO / GRPO —— LLM RL 训练核心算法> 从零用 PyTorch 实现 PPO 和 GRPO，理解 DeepSeek-R1 / verl 背后的算法。

## Part 1: PPO (Proximal Policy Optimization)### 核心公式$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min(r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t) \right]$$其中 $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


### 1.1 简化版 Policy Network

In [ ]:
class SimplePolicy(nn.Module):
    """简化版 policy network：输入 state，输出 action logits."""
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        return self.net(x)

    def get_action(self, state):
        logits = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action)

    def evaluate(self, states, actions):
        logits = self.forward(states)
        dist = Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy()

class ValueNetwork(nn.Module):
    """Critic: 估计 state value."""
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

print("✅ Policy & Value networks defined")


### 1.2 GAE (Generalized Advantage Estimation)

In [ ]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """计算 GAE advantages 和 returns.
    
    GAE(γ,λ): Â_t = Σ_{l=0}^{T-t} (γλ)^l δ_{t+l}
    其中 δ_t = r_t + γV(s_{t+1}) - V(s_t)
    """
    advantages = torch.zeros_like(rewards)
    last_gae = 0
    T = len(rewards)
    for t in reversed(range(T)):
        next_value = values[t + 1] if t + 1 < T else 0
        next_done = dones[t + 1] if t + 1 < T else 1
        delta = rewards[t] + gamma * next_value * (1 - next_done) - values[t]
        advantages[t] = last_gae = delta + gamma * lam * (1 - next_done) * last_gae
    returns = advantages + values[:T]
    return advantages, returns

# ---- 测试 GAE ----
rewards = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])
values = torch.tensor([0.5, 0.4, 0.6, 0.3, 0.5])
dones = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])
adv, ret = compute_gae(rewards, values, dones)
print(f"Advantages: {adv.numpy().round(3)}")
print(f"Returns:    {ret.numpy().round(3)}")
print("✅ GAE computation works")


### 1.3 PPO 训练循环

In [ ]:
def ppo_update(policy, value_net, optimizer_p, optimizer_v,
               states, actions, old_log_probs, advantages, returns,
               clip_eps=0.2, epochs=4, batch_size=64):
    """PPO clipped objective + value loss."""
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    losses = {'policy': [], 'value': [], 'entropy': []}
    n = len(states)
    
    for _ in range(epochs):
        idx = torch.randperm(n)
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            mb = idx[start:end]
            
            # Policy loss (clipped)
            new_log_probs, entropy = policy.evaluate(states[mb], actions[mb])
            ratio = torch.exp(new_log_probs - old_log_probs[mb])
            surr1 = ratio * advantages[mb]
            surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages[mb]
            policy_loss = -torch.min(surr1, surr2).mean()
            entropy_loss = -entropy.mean()
            
            optimizer_p.zero_grad()
            (policy_loss + 0.01 * entropy_loss).backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer_p.step()
            
            # Value loss
            value_pred = value_net(states[mb])
            value_loss = F.mse_loss(value_pred, returns[mb])
            
            optimizer_v.zero_grad()
            value_loss.backward()
            nn.utils.clip_grad_norm_(value_net.parameters(), 0.5)
            optimizer_v.step()
            
            losses['policy'].append(policy_loss.item())
            losses['value'].append(value_loss.item())
            losses['entropy'].append(-entropy_loss.item())
    
    return {k: np.mean(v) for k, v in losses.items()}

print("✅ PPO update function defined")


### 1.4 模拟训练：简单环境

In [ ]:
class SimpleBandit:
    """简化环境：模拟 LLM reward optimization.
    State: 随机 embedding
    Action: 选择 response 策略 (0..3)
    Reward: action 2 最优，模拟 rule-based reward
    """
    def __init__(self, state_dim=8):
        self.state_dim = state_dim

    def reset(self):
        return torch.randn(self.state_dim)

    def step(self, state, action):
        # 模拟 rule-based reward (DeepSeek-R1 style)
        base_reward = -0.5
        if action == 2:
            base_reward = 1.0  # 正确答案
        elif action == 1:
            base_reward = 0.3  # 部分正确
        noise = torch.randn(1).item() * 0.1
        return base_reward + noise

# 训练循环
env = SimpleBandit()
policy = SimplePolicy(state_dim=8, action_dim=4)
value_net = ValueNetwork(state_dim=8)
opt_p = torch.optim.Adam(policy.parameters(), lr=3e-4)
opt_v = torch.optim.Adam(value_net.parameters(), lr=1e-3)

episode_rewards = []
for episode in range(200):
    states, actions, log_probs, rewards, dones = [], [], [], [], []
    
    for _ in range(64):  # collect rollout
        state = env.reset()
        action, lp = policy.get_action(state)
        reward = env.step(state, action.item())
        states.append(state)
        actions.append(action)
        log_probs.append(lp)
        rewards.append(reward)
        dones.append(0.0)
    dones[-1] = 1.0
    
    states_t = torch.stack(states)
    actions_t = torch.stack(actions)
    log_probs_t = torch.stack(log_probs).detach()
    rewards_t = torch.tensor(rewards)
    dones_t = torch.tensor(dones)
    
    with torch.no_grad():
        values = value_net(states_t)
    
    advantages, returns = compute_gae(rewards_t, values, dones_t)
    info = ppo_update(policy, value_net, opt_p, opt_v,
                      states_t, actions_t, log_probs_t, advantages, returns)
    episode_rewards.append(rewards_t.mean().item())

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(episode_rewards, alpha=0.3)
ax1.plot(np.convolve(episode_rewards, np.ones(20)/20, mode='valid'), linewidth=2)
ax1.set_xlabel("Episode"); ax1.set_ylabel("Mean Reward")
ax1.set_title("PPO Training Curve")
ax1.axhline(y=1.0, color='r', linestyle='--', label='optimal')
ax1.legend()

# Action distribution
with torch.no_grad():
    test_states = torch.stack([env.reset() for _ in range(1000)])
    logits = policy(test_states)
    probs = F.softmax(logits, dim=-1).mean(0)
ax2.bar(range(4), probs.numpy())
ax2.set_xlabel("Action"); ax2.set_ylabel("Probability")
ax2.set_title("Learned Policy Distribution")
ax2.set_xticks(range(4), ["bad(-0.5)", "partial(0.3)", "optimal(1.0)", "bad(-0.5)"])
plt.tight_layout()
plt.savefig("ppo_training.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"\n最终策略倾向 action=2 (最优): prob={probs[2]:.3f}")
print("✅ PPO training complete!")


---## Part 2: GRPO (Group Relative Policy Optimization)### DeepSeek-R1 的核心算法**与 PPO 的关键区别**：- **无 Critic**：不需要 Value Network- **组内相对优势**：对同一 prompt 生成 G 个 response，用组内相对 reward 作为 advantage- **KL 惩罚**：直接对比新旧策略的 KL 散度$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_1,...,r_G\})}{\text{std}(\{r_1,...,r_G\})}$$

In [ ]:
class GRPO:
    """Group Relative Policy Optimization.
    
    核心思想：
    1. 对每个 prompt，用旧策略生成 G 个 response
    2. 计算每个 response 的 reward
    3. 组内标准化 reward 作为 advantage
    4. PPO-clip + KL penalty 更新策略
    """
    def __init__(self, policy, ref_policy, lr=1e-4, 
                 clip_eps=0.2, kl_coef=0.1, group_size=4):
        self.policy = policy
        self.ref_policy = ref_policy  # frozen reference policy
        self.optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
        self.clip_eps = clip_eps
        self.kl_coef = kl_coef
        self.group_size = group_size

    def compute_group_advantage(self, rewards):
        """组内标准化 reward → advantage.
        rewards: (num_prompts, group_size)
        """
        mean = rewards.mean(dim=1, keepdim=True)
        std = rewards.std(dim=1, keepdim=True) + 1e-8
        return (rewards - mean) / std

    def update(self, prompts, all_actions, all_old_log_probs, rewards):
        """
        prompts: (num_prompts, state_dim)
        all_actions: (num_prompts, group_size)
        all_old_log_probs: (num_prompts, group_size)
        rewards: (num_prompts, group_size)
        """
        advantages = self.compute_group_advantage(rewards)  # (P, G)
        
        total_loss = 0
        P, G = rewards.shape
        
        for p in range(P):
            state = prompts[p].unsqueeze(0).expand(G, -1)  # (G, dim)
            
            # New policy log probs
            new_lp, entropy = self.policy.evaluate(state, all_actions[p])
            
            # Reference policy log probs (for KL)
            with torch.no_grad():
                ref_lp, _ = self.ref_policy.evaluate(state, all_actions[p])
            
            # PPO-clip objective
            ratio = torch.exp(new_lp - all_old_log_probs[p])
            adv = advantages[p]
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1-self.clip_eps, 1+self.clip_eps) * adv
            clip_loss = -torch.min(surr1, surr2).mean()
            
            # KL penalty (approximate)
            kl = (all_old_log_probs[p] - new_lp).mean()
            
            total_loss += clip_loss + self.kl_coef * kl - 0.01 * entropy.mean()
        
        total_loss /= P
        self.optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        
        return total_loss.item()

print("✅ GRPO class defined")


### 2.1 GRPO 模拟训练

In [ ]:
import copy

# 环境：模拟 math reward (rule-based, DeepSeek-R1 style)
def math_reward_fn(action):
    """模拟 rule-based reward: action=2 正确，其他错误."""
    if action == 2:
        return 1.0
    elif action == 1:
        return 0.3
    else:
        return -0.5

env = SimpleBandit()
policy = SimplePolicy(state_dim=8, action_dim=4)
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False

grpo = GRPO(policy, ref_policy, lr=3e-4, group_size=8, kl_coef=0.05)

G = 8  # group size
grpo_rewards = []

for step in range(300):
    # 1. 采集一批 prompts
    num_prompts = 16
    prompts = torch.stack([env.reset() for _ in range(num_prompts)])
    
    # 2. 对每个 prompt 生成 G 个 response
    all_actions = torch.zeros(num_prompts, G, dtype=torch.long)
    all_log_probs = torch.zeros(num_prompts, G)
    rewards = torch.zeros(num_prompts, G)
    
    with torch.no_grad():
        for p in range(num_prompts):
            state = prompts[p].unsqueeze(0).expand(G, -1)
            for g in range(G):
                a, lp = policy.get_action(prompts[p].unsqueeze(0))
                all_actions[p, g] = a
                all_log_probs[p, g] = lp
                rewards[p, g] = math_reward_fn(a.item())
    
    # 3. GRPO update
    loss = grpo.update(prompts, all_actions, all_log_probs, rewards)
    grpo_rewards.append(rewards.mean().item())

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(grpo_rewards, alpha=0.3, label='raw')
ax1.plot(np.convolve(grpo_rewards, np.ones(20)/20, mode='valid'), linewidth=2, label='smooth')
ax1.set_xlabel("Step"); ax1.set_ylabel("Mean Reward")
ax1.set_title("GRPO Training Curve (No Critic!)")
ax1.axhline(y=1.0, color='r', linestyle='--', label='optimal')
ax1.legend()

with torch.no_grad():
    test_states = torch.stack([env.reset() for _ in range(1000)])
    logits = policy(test_states)
    probs = F.softmax(logits, dim=-1).mean(0)
ax2.bar(range(4), probs.numpy())
ax2.set_xlabel("Action"); ax2.set_ylabel("Probability")
ax2.set_title("GRPO Learned Distribution")
ax2.set_xticks(range(4), ["bad", "partial", "optimal", "bad"])
plt.tight_layout()
plt.savefig("grpo_training.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"\nGRPO 最终 action=2 概率: {probs[2]:.3f}")
print("✅ GRPO training complete!")


---## Part 3: PPO vs GRPO 对比总结| 维度 | PPO | GRPO ||------|-----|------|| **Critic** | 需要 Value Network | ❌ 不需要 || **Advantage** | GAE (TD-based) | 组内相对标准化 || **KL 控制** | clip only | clip + KL penalty || **生成量** | 1 response/prompt | G responses/prompt || **适用** | 通用 RL | 专为 LLM 设计 || **论文** | Schulman et al. 2017 | DeepSeek-R1 (2025) |### 面试回答模板```"PPO 需要训练一个 Critic 来估计 Value，计算 GAE advantage。 GRPO 去掉了 Critic，改为对同一个 prompt 生成 G 个 response， 用组内 reward 的 z-score 作为 advantage。 好处是：(1) 省掉 Critic 的显存和训练开销， (2) 天然适合 LLM 的 rule-based reward（数学题对错）， (3) DeepSeek-R1 用 GRPO 从零训出了 reasoning 能力。"```